# Data Audit

## Business Objective

Before starting exploratory analysis, it is necessary to validate the quality of the source data and determine how missing values, inconsistent records and data quality issues should be handled.


## Notebook Objectives

- Assess missing values
- Evaluate duplicate records
- Validate business rules
- Verify referential integrity
- Decide how each issue should be handled
- Prepare clean datasets for analysis

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

from IPython.display import display

In [3]:
DATA_PATH = Path("../06_Datasets/raw")

dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "categories": "product_category_name_translation.csv"
}

datasets = {}
for name, filename in dataset_files.items():
    datasets[name] = pd.read_csv(DATA_PATH / filename)

## Missing Values Analysis

Missing values are not always data errors.

This section identifies missing values for every dataset and evaluates whether they require further action.

In [6]:
missing_reports = {}

for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        report = pd.DataFrame({
            "Missing Values": missing,
            "Missing %": (
                missing / len(df) * 100
            ).round(2)
        })
        missing_reports[name] = report

In [9]:
for name, report in missing_reports.items():
    print("=" * 80)
    print(name.upper())
    print("=" * 80)
    display(report)

ORDERS


,Missing Values,Missing %
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


ORDER_REVIEWS


,Missing Values,Missing %
review_comment_title,87656,88.34
review_comment_message,58247,58.70


PRODUCTS


,Missing Values,Missing %
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [22]:
missing_products = datasets["products"][
    datasets["products"]["product_category_name"].isna()
]

missing_products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [23]:
missing_products.nunique()

product_id                    610
product_category_name           0
product_name_lenght             0
product_description_lenght      0
product_photos_qty              0
product_weight_g              145
product_length_cm              54
product_height_cm              49
product_width_cm               47
dtype: int64

In [24]:
datasets["products"][
    datasets["products"]["product_weight_g"].isna()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Orders, Order_Reviews, Products

Observation

Several attributes contain missing values.

Possible Reasons

- incomplete catalog information;
- optional product characteristics;
- missing data during collection.

Preliminary Decision

Do not fill missing values until the business purpose of these attributes is determined.

### Orders

Observation

Three delivery-related datetime columns contain missing values.

Possible Reasons

- cancelled orders;
- orders still in processing;
- incomplete delivery lifecycle.

Preliminary Decision

Do not impute missing values.

Further investigation of the order status is required before any cleaning decisions are made.

### Order Reviews

Observation
Review titles and review messages contain a high proportion of missing values.

Possible Reasons
Customers are not required to leave written feedback when submitting a review.

Preliminary Decision
Missing values will be preserved.

No imputation is required because the absence of a review message represents valid customer behavior rather than missing information.

### Products

Observation

Product metadata fields share the same number of missing values, suggesting that missing information is concentrated within the same product records.

Only two products have missing physical dimensions.

Possible Reasons

- incomplete product registration;
- missing catalog information.

Preliminary Decision

Further investigation is required.

No missing values will be filled before identifying the affected products.

In [10]:
datasets["geolocation"].duplicated().sum()

np.int64(261831)

In [11]:
datasets["geolocation"]["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(981148)

In [16]:
(
    datasets["geolocation"]
    .groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
dtype: int64

### Geolocation

Observation

The dataset contains 261831 duplicated rows.

Possible Reasons

- repeated geographic records;
- duplicated source data;
- multiple identical entries in the reference table.

Preliminary Decision

Further investigation is required before removing duplicate records.

The duplication pattern must be analyzed to determine whether these records are redundant or represent valid geographical information.

In [9]:

duplicates = datasets["geolocation"][datasets["geolocation"].duplicated(keep=False)]
len(duplicates)
#display(duplicates.head(20))

390005

In [8]:
datasets["geolocation"].nunique()

geolocation_zip_code_prefix     19015
geolocation_lat                717360
geolocation_lng                717613
geolocation_city                 8011
geolocation_state                  27
dtype: int64

In [14]:
duplicates = datasets["geolocation"][
    datasets["geolocation"].duplicated(
        subset=[
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ],
        keep=False
    )
]

display(duplicates.head(10))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
14,1037,-23.545187,-46.637855,são paulo,SP


In [15]:
full_duplicates = datasets["geolocation"].duplicated().sum()
print(full_duplicates)

261831


In [16]:
datasets["geolocation"]["geolocation_zip_code_prefix"].nunique()

19015

In [17]:
zip_counts = (
    datasets["geolocation"]
    .groupby("geolocation_zip_code_prefix")
    .size()
    .sort_values(ascending=False)
)

zip_counts.head(10)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
dtype: int64

In [18]:
datasets["geolocation"][
    datasets["geolocation"]["geolocation_zip_code_prefix"] == 1046
]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP
19,1046,-23.545884,-46.643163,sao paulo,SP
20,1046,-23.545166,-46.643786,sao paulo,SP
...,...,...,...,...,...
1432,1046,-23.545166,-46.643786,sao paulo,SP
1437,1046,-23.547191,-46.642923,sao paulo,SP
1439,1046,-23.545158,-46.643880,sao paulo,SP
1450,1046,-23.545292,-46.643130,sao paulo,SP


In [21]:
duplicates = datasets["geolocation"][
    datasets["geolocation"].duplicated(keep=False)
]
duplicates.head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [20]:
before = len(datasets["geolocation"])
after = len(datasets["geolocation"].drop_duplicates())

print(f"Before: {before:,}")
print(f"After : {after:,}")
print(f"Removed: {before - after:,}")

Before: 1,000,163
After : 738,332
Removed: 261,831


### Geolocation

**Observation**

The dataset contains 261831 fully duplicated records.

In addition, multiple observations exist for the same ZIP code prefix, but many of them have different latitude and longitude values.

**Interpretation**

Repeated ZIP code prefixes are expected because a single postal area may contain multiple geographic points.

However, fully identical records do not add new information and are likely redundant.

**Preliminary Decision**

- Preserve multiple coordinates for the same ZIP code.
- Remove only fully duplicated rows after confirming that no business information is lost.

# Audit Conclusions

## Summary

The data quality assessment identified several issues requiring attention before exploratory analysis.

### Key Findings

- Missing delivery dates in the Orders table are consistent with the order lifecycle and should not be imputed.
- Missing review titles and comments represent valid customer behavior.
- Missing product metadata requires further investigation but should remain unchanged at this stage.
- The Geolocation table contains fully duplicated records, while multiple coordinates for the same ZIP code are considered valid.

## Overall Assessment

The dataset is suitable for business analysis after minor preprocessing steps, including duplicate removal and datatype conversion.

| Dataset       | Issue                          | Decision    | Status |
| ------------- | ------------------------------ | ----------- | ------ |
| Orders        | Missing delivery dates         | Keep        | ✅      |
| Order Reviews | Missing comments               | Keep        | ✅      |
| Products      | Missing product metadata       | Investigate | 🟡     |
| Products      | Missing dimensions (2 records) | Investigate | 🟡     |
| Geolocation   | Full duplicate rows            | Remove      | ✅      |
